# 4 · Predictive Modelling — Saturation Forecasting

This notebook implements the predictive modeling component of the **Geospatial Repletion & Saturation Modelling** project. 
The goal is to build machine learning models using **Apache Spark MLlib** to forecast localized infrastructure saturation 30–60 minutes in advance, operating on simulated citizen traffic flows. 

## Constraints:
1. **Strict Spark MLlib**: All data manipulation and modeling are done using PySpark SQL and MLlib. No `pandas` or `scikit-learn` are used to respect big data processing standards.
2. **No Data Leakage**: We use a temporal split instead of a random split, and our target is a future lead variable (`lead(count, k)`), ensuring the model is forecasting rather than fitting a tautology.

In [1]:
import os
import sys
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# Find project root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

# Setup native Hadoop binaries on Windows
import sys; sys.path.append(os.path.abspath('..'))
from src.step_08_bootstrapping import setup_winutils
setup_winutils(PROJECT_ROOT)

# Initialize local Spark Session
spark = (
    SparkSession.builder
    .appName("Saturation-Forecasting-Engine")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "10")
    .getOrCreate()
)

print(f"SparkSession started successfully. Spark version: {spark.version}")

2026-07-08 13:37:06,892 - INFO - Hadoop environment path configuration active: HADOOP_HOME=C:\Users\fedka\Documents\GitHub\Geospatial Repletion & Saturation Modelling\data\winutils


SparkSession started successfully. Spark version: 3.5.8


## 4.1 Auto-Load Verification & Configuration

To save computation time during reviews and grading, we check if a pre-trained model and pre-aggregated infrastructure count files exist on disk. If they do, we can load them directly. Setting `FORCE_RETRAIN = True` will force the notebook to rerun the entire feature engineering and training pipeline.

In [2]:
FORCE_RETRAIN = False

model_path = PROJECT_ROOT / "models" / "gbt_saturation_forecaster"
data_path = PROJECT_ROOT / "data" / "geospatial_output" / "infrastructure_activity_counts.csv"

has_model = model_path.exists()
has_data = data_path.exists()

print(f"Pre-aggregated data found: {has_data}")
print(f"Pre-trained GBT model found: {has_model}")
if not FORCE_RETRAIN and has_model and has_data:
    print("\n>>> PRE-TRAINED MODEL READY. Pipelines can be skipped or loaded directly.")
else:
    print("\n>>> PIPELINE WILL TRAIN A NEW MODEL.")

Pre-aggregated data found: True
Pre-trained GBT model found: True

>>> PRE-TRAINED MODEL READY. Pipelines can be skipped or loaded directly.


## 4.2 Data Ingestion & Target Definition

First, we load the aggregated infrastructure counts from the geospatial pipeline output. We define the infrastructure capacities according to Vienna's guidelines:
- Bike Path: 3,000 users/hour capacity
- Pedestrian Zone: 600 users/hour capacity
- Default (Other): 1,000 users/hour capacity

Our predictive model will forecast saturation indexes ($s_{t+\Delta}$) rather than immediate counts to provide early-warning bottlenecks.

In [3]:
if not has_data:
    # Generate dummy data to ensure no crash if running first time without geospatial step
    print("Creating mock activity counts data...")
    mock_data = []
    for i in range(100):
        mock_data.append(("1., Stephansplatz", "pedestrian_zone", "walk", i, 100 + (i % 5) * 50))
        mock_data.append(("Markierte Anlagen", "bike_path", "bike", i, 200 + (i % 3) * 150))
    df_raw = spark.createDataFrame(mock_data, ["infrastructure_label", "infrastructure_type", "Activity", "time_window_index", "count"])
else:
    df_raw = spark.read.csv(str(data_path), header=True, inferSchema=True)

# Apply capacity scaling
df_capped = df_raw.withColumn(
    "max_capacity",
    F.when(F.col("infrastructure_type") == "bike_path", 3000.0)
     .when(F.col("infrastructure_type") == "pedestrian_zone", 600.0)
     .otherwise(1000.0)
)

df_capped = df_capped.withColumn(
    "saturation_index",
    F.col("count").cast("double") / F.col("max_capacity")
)

df_capped.show(5)

+--------------------+-------------------+--------+-----------------+-----+------------+-------------------+
|infrastructure_label|infrastructure_type|Activity|time_window_index|count|max_capacity|   saturation_index|
+--------------------+-------------------+--------+-----------------+-----+------------+-------------------+
|   Markierte Anlagen|          bike_path|    bike|                0|10360|      3000.0|  3.453333333333333|
|   Getrennte Führung|          bike_path|    bike|                0|11380|      3000.0| 3.7933333333333334|
|            Radroute|          bike_path|    bike|                0| 5810|      3000.0| 1.9366666666666668|
|          unassigned|          bike_path|    bike|                0|  220|      3000.0|0.07333333333333333|
|   Getrennte Führung|          bike_path|    bike|                1|13120|      3000.0|  4.373333333333333|
+--------------------+-------------------+--------+-----------------+-----+------------+-------------------+
only showing top 5 

## 4.3 Feature Engineering (Spark SQL Windowing)

To enable forecasting, we build temporal lags and statistics over the segments using Spark analytical windows. We extract:
- Lags 1 through 6 (`count_lag_1` to `count_lag_6`)
- 3-window rolling average (`count_rolling_mean_3`)
- 3-window rolling standard deviation (`count_rolling_std_3`)
- First-order count difference (`count_delta`)
- Target lead count (`target_count` = value at $t + 5$ windows ahead)

In [4]:
# Window specifications
w_seg = Window.partitionBy("infrastructure_label", "infrastructure_type").orderBy("time_window_index")
w_roll = w_seg.rowsBetween(-3, -1)

df_features = df_capped

# 1. Lags
for lag_idx in range(1, 7):
    df_features = df_features.withColumn(f"count_lag_{lag_idx}", F.lag("count", lag_idx).over(w_seg))

# 2. Rolling stats
df_features = df_features.withColumn("count_rolling_mean_3", F.avg("count").over(w_roll))
df_features = df_features.withColumn("count_rolling_std_3", F.stddev("count").over(w_roll))

# 3. Delta
df_features = df_features.withColumn("count_delta", F.col("count") - F.col("count_lag_1"))

# 4. Target variable: Saturation at t + 5 windows ahead (Lead)
forecast_lead = 5
df_features = df_features.withColumn("target_count", F.lead("count", forecast_lead).over(w_seg))
df_features = df_features.withColumn("target_saturation", F.col("target_count").cast("double") / F.col("max_capacity"))

# Remove null values created by lead/lag offsets
df_clean = df_features.na.drop()

# Encode infrastructure type categories
indexer = StringIndexer(inputCol="infrastructure_type", outputCol="infra_type_idx", handleInvalid="keep")
df_indexed = indexer.fit(df_clean).transform(df_clean)

# Assemble into PySpark features vector
feature_cols = [
    "infra_type_idx", "count",
    "count_lag_1", "count_lag_2", "count_lag_3",
    "count_lag_4", "count_lag_5", "count_lag_6",
    "count_rolling_mean_3", "count_rolling_std_3", "count_delta"
]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
ml_df = assembler.transform(df_indexed).select("time_window_index", "features", "target_saturation").withColumnRenamed("target_saturation", "label")

ml_df.show(3, truncate=False)

+-----------------+---------------------------------------------------------------------------------------------------+-------------------+
|time_window_index|features                                                                                           |label              |
+-----------------+---------------------------------------------------------------------------------------------------+-------------------+
|6                |[2.0,4640.0,5340.0,4610.0,4180.0,3690.0,2050.0,750.0,4710.0,586.4298764558299,-700.0]              |0.23333333333333334|
|7                |[2.0,2010.0,4640.0,5340.0,4610.0,4180.0,3690.0,2050.0,4863.333333333333,413.07787804884117,-2630.0]|0.5833333333333334 |
|8                |[2.0,550.0,2010.0,4640.0,5340.0,4610.0,4180.0,3690.0,3996.6666666666665,1755.7429576487934,-1460.0]|1.0166666666666666 |
+-----------------+---------------------------------------------------------------------------------------------------+-------------------+
only showing top 3 r

## 4.4 Temporal Train/Test Split

In time series modeling, a random train/test split causes severe data leakage (future records predicting past records). We split the data chronologically: the first 70% of time windows for training, and the remaining 30% for evaluation.

In [5]:
# Find max index to split
max_idx_row = ml_df.agg(F.max("time_window_index")).collect()[0]
max_idx = max_idx_row[0] if max_idx_row[0] is not None else 0
split_idx = int(max_idx * 0.7)

train_df = ml_df.filter(F.col("time_window_index") <= split_idx).cache()
test_df = ml_df.filter(F.col("time_window_index") > split_idx).cache()

print(f"Training rows: {train_df.count()}")
print(f"Evaluation rows: {test_df.count()}")

Training rows: 1492


Evaluation rows: 694


## 4.5 Model Comparison & Training (LR → RF → GBT)

We evaluate three regression algorithms of increasing mathematical complexity:
1. **Linear Regression**: A baseline model mapping linear mappings of features.
2. **Random Forest Regressor**: Ensembles independent decision trees to capture non-linear relationships.
3. **GBT Regressor (Gradient Boosted Trees)**: Iteratively minimizes training loss using functional gradient descent, optimizing weights to reduce residuals. Shrinkage acts as a regularizer to prevent overfitting.

In [6]:
eval_rmse = RegressionEvaluator(metricName="rmse")
eval_r2 = RegressionEvaluator(metricName="r2")
eval_mae = RegressionEvaluator(metricName="mae")

# 1. Linear Regression Baseline
print("Training Linear Regression model...")
lr = LinearRegression(featuresCol="features", labelCol="label", regParam=0.05)
lr_model = lr.fit(train_df)
lr_preds = lr_model.transform(test_df)

# 2. Random Forest Regressor
print("Training Random Forest Regressor...")
rf = RandomForestRegressor(featuresCol="features", labelCol="label", numTrees=50, maxDepth=6, seed=42)
rf_model = rf.fit(train_df)
rf_preds = rf_model.transform(test_df)

# 3. Gradient Boosted Trees (GBT)
print("Training GBT Regressor...")
gbt = GBTRegressor(featuresCol="features", labelCol="label", maxIter=80, maxDepth=5, stepSize=0.1, seed=42)
gbt_model = gbt.fit(train_df)
gbt_preds = gbt_model.transform(test_df)

print("Training complete.")

Training Linear Regression model...


Training Random Forest Regressor...


Training GBT Regressor...


Training complete.


## 4.6 Model Performance Metrics

We compare the evaluation scores on the future test set.

In [7]:
models_list = {
    "Linear Regression": lr_preds,
    "Random Forest": rf_preds,
    "GBT Regressor": gbt_preds
}

for name, preds in models_list.items():
    rmse = eval_rmse.evaluate(preds)
    r2 = eval_r2.evaluate(preds)
    mae = eval_mae.evaluate(preds)
    print(f"{name:20s} -> RMSE: {rmse:.4f} | R²: {r2:.4f} | MAE: {mae:.4f}")

Linear Regression    -> RMSE: 3.4607 | R²: 0.8418 | MAE: 1.5489


Random Forest        -> RMSE: 1.4299 | R²: 0.9730 | MAE: 0.3193


GBT Regressor        -> RMSE: 1.4408 | R²: 0.9726 | MAE: 0.3195


## 4.7 Model Diagnostics & Feature Importances

We extract Gini importances from the GBT Regressor to identify which time lags contribute most to the prediction.

In [8]:
importances = gbt_model.featureImportances
print("GBT Feature Importances:")
for idx, name in enumerate(feature_cols):
    print(f"  {name:25s}: {importances[idx]:.4f}")

# Save trained GBT model weights for future load
model_save_dir = str(PROJECT_ROOT / "models" / "gbt_saturation_forecaster")
gbt_model.write().overwrite().save(model_save_dir)
print(f"\nSaved model weights to: {model_save_dir}")

GBT Feature Importances:
  infra_type_idx           : 0.0151
  count                    : 0.9282
  count_lag_1              : 0.0016
  count_lag_2              : 0.0015
  count_lag_3              : 0.0017
  count_lag_4              : 0.0029
  count_lag_5              : 0.0013
  count_lag_6              : 0.0038
  count_rolling_mean_3     : 0.0331
  count_rolling_std_3      : 0.0040
  count_delta              : 0.0068



Saved model weights to: C:\Users\fedka\Documents\GitHub\Geospatial Repletion & Saturation Modelling\models\gbt_saturation_forecaster


## 4.8 Pre-trained Model Load Demo

Below we demonstrate how to load the GBT model directly from disk, bypassing the training phase entirely. This can be used for rapid deployments.

In [9]:
from pyspark.ml.regression import GBTRegressionModel

print("Loading saved model weights...")
loaded_gbt = GBTRegressionModel.load(str(PROJECT_ROOT / "models" / "gbt_saturation_forecaster"))
new_predictions = loaded_gbt.transform(test_df)

print("Evaluation on loaded GBT model:")
print(f"  Loaded Model RMSE: {eval_rmse.evaluate(new_predictions):.4f}")

print("\nSample predictions (Lead Saturation Index):")
new_predictions.select("features", "label", "prediction").show(5, truncate=False)

Loading saved model weights...


Evaluation on loaded GBT model:
  Loaded Model RMSE: 1.4408

Sample predictions (Lead Saturation Index):


+-------------------------------------------------------------------------------------------------+-----+------------------+
|features                                                                                         |label|prediction        |
+-------------------------------------------------------------------------------------------------+-----+------------------+
|[0.0,1380.0,910.0,1320.0,1200.0,990.0,1280.0,1280.0,1143.3333333333333,210.79215671683167,470.0] |1.2  |1.3712628643039293|
|[0.0,1390.0,1380.0,910.0,1320.0,1200.0,990.0,1280.0,1203.3333333333333,255.79940057266228,10.0]  |0.98 |1.1961358833305147|
|[0.0,1090.0,1390.0,1380.0,910.0,1320.0,1200.0,990.0,1226.6666666666667,274.2869543622761,-300.0] |1.09 |1.213383830833358 |
|[0.0,1200.0,1090.0,1390.0,1380.0,910.0,1320.0,1200.0,1286.6666666666667,170.39170558842744,110.0]|1.16 |1.1637312571640328|
|[0.0,1350.0,1200.0,1090.0,1390.0,1380.0,910.0,1320.0,1226.6666666666667,151.7673658377628,150.0] |1.26 |1.2026510923132887|
